# Local RAG Test Notebook

This notebook demonstrates a small retrieval-augmented generation workflow using the local Chroma service and Ollama model server.

In [1]:
import os
import requests

CHROMA_URL = os.getenv('CHROMA_URL', 'http://localhost:8001')
OLLAMA_URL = os.getenv('OLLAMA_URL', 'http://localhost:11434')
MODEL = os.getenv('DEFAULT_MODEL', 'llama3.2:1b')

print('Chroma URL:', CHROMA_URL)
print('Ollama URL:', OLLAMA_URL)
print('Model:', MODEL)

Chroma URL: http://localhost:8001
Ollama URL: http://localhost:11434
Model: llama3.2:1b


In [2]:
docs = [
    {'id': 'doc-01', 'text': 'Ollama runs locally and exposes a standard completions API compatible with local clients.', 'metadata': {'topic': 'infrastructure'}},
    {'id': 'doc-02', 'text': 'Chroma stores vectors and can retrieve similar documents using semantic search.', 'metadata': {'topic': 'rag'}},
    {'id': 'doc-03', 'text': 'A RAG pipeline combines retrieved documents with a prompt to improve model grounding.', 'metadata': {'topic': 'security'}},
]

response = requests.post(f'{CHROMA_URL}/ingest', json=docs, timeout=30)
response.raise_for_status()
print('Ingest response:', response.json())

Ingest response: {'ingested': 3, 'ids': ['doc-01', 'doc-02', 'doc-03']}


In [3]:
query_payload = {'query': 'How does this local stack use Chroma for retrieval?', 'top_k': 3}
response = requests.post(f'{CHROMA_URL}/query', json=query_payload, timeout=30)
response.raise_for_status()
results = response.json().get('results', [])
print('Top retrieved documents:')
for doc in results:
    print(f"- {doc['id']}: {doc['text']} (distance={doc['distance']})")

Top retrieved documents:
- doc-02: Chroma stores vectors and can retrieve similar documents using semantic search. (distance=0.9278044267550755)
- doc-01: Ollama runs locally and exposes a standard completions API compatible with local clients. (distance=1.4018787274098639)
- doc-03: A RAG pipeline combines retrieved documents with a prompt to improve model grounding. (distance=1.4424661652757793)


In [4]:
retrieved_text = '\n\n'.join([doc['text'] for doc in results])
prompt = f'''You are a local assistant. Use the retrieved documents below to answer the question.

Documents:
{retrieved_text}

Question: How is Chroma used in this stack?
Answer concisely and cite the relevant document segments.'''

payload = {
    'model': MODEL,
    'messages': [{'role': 'user', 'content': prompt}],
    'max_tokens': 200,
    'temperature': 0.1,
}
ollama_response = requests.post(f'{OLLAMA_URL}/v1/chat/completions', json=payload, timeout=120)
ollama_response.raise_for_status()
answer = ollama_response.json().get('choices', [{}])[0].get('message', {}).get('content', '')
print('Model answer:')
print(answer)

Model answer:
Based on the provided documents, Chroma is used as follows:

1. **Semantic search**: Chroma can retrieve similar documents using semantic search (document segment not explicitly mentioned, but implied by the mention of "similar documents").
2. **Local client support**: Ollama exposes a standard completions API compatible with local clients, indicating that Chroma can be integrated with local assistants (document segment).
3. **RAG pipeline**: A RAG (Recommendation and Alignment) pipeline is used to combine retrieved documents with a prompt to improve model grounding (document segment).
